<a href="https://colab.research.google.com/github/AbdulRehman6162/Rest_Universe_Scrapper/blob/master/Profiling_GoogleSheets_Drive_V2_3_1_Fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Restaurant Profiling — Google Maps Collector V2.3.3

Google Sheets is the system of record (`Restaurant Master`, `ISB-RWP Target Areas`).
Google Drive stores resumable checkpoints, append-only logs, timestamped raw observations,
and a master snapshot before each write. No Excel upload is required.

Run cells in order in Google Colab. Set the Google Sheet name in Cell 3 and output folder
and run limits in Cell 4. Defaults run 10 refreshes and 2 searches of up to 10 results.
Inspect the diagnostics before running the writer cell. Run only one notebook against a
master at a time; Sheets does not provide a transaction lock for concurrent collectors.

`RUN_ID` defaults to today's UTC date: rerunning that day resumes successful observations
less than 24 hours old. Set a new `RUN_ID` for an immediate fresh collection. Errors are
retried. Old checkpoint formats are not reused. Each record carries `Last Observed At`;
`Date Profiled` remains the original profiling date.

A known Place ID matches only that Place ID. Different branches sharing a phone remain
separate. Ambiguous fallback keys do not choose a row. Existing duplicate/invalid internal
IDs or duplicate Place IDs stop writes for manual identity resolution. Missing IDs are
allocated above the existing maximum; existing IDs are never renumbered.

Services: **Y = explicit positive evidence; blank = unknown**. Incomplete hours stay blank.
CRM `Status`, analyst `Notes`, cuisines, and external enrichment are preserved. Current
opening status and business status are separate fields. Dashboard repair is opt-in.

This freezes the Google identity layer. Google Places API, menu source discovery, price
history, competitor matching, and owner portal remain separate future pipelines.
Live Maps UI and authenticated Sheets integration must be smoke-tested in Colab.


### If SEARCH finds zero links

This build waits for actual listing evidence, supports relative/CID/place-ID listing URLs,
and distinguishes confirmed no-results from blocked, consent, HTTP, JavaScript, and
unrecognized pages. It stops on discovery failure rather than announcing success.
The failure screenshot is displayed in Colab and saved with HTML and a JSON report in
`OUTPUT_DIR/search_debug`. Share the displayed screenshot/report to diagnose the exact
response in your Colab session. Do not repeatedly retry a blocked session.

`Existing Master rows: 0` means the selected sheet has no data rows; that only skips
REFRESH and does not prevent SEARCH. Confirm the spreadsheet name/URL in Cell 3 if you
expected existing restaurants. After updating this notebook, restart the Colab session
and run cells in order so the revised helper functions replace older definitions.


In [ ]:
# CELL 1 — Install dependencies
%pip -q install playwright openpyxl pandas nest_asyncio gspread google-auth
import subprocess
import sys
subprocess.run([sys.executable, "-m", "playwright", "install", "--with-deps", "chromium"], check=True)


In [ ]:

# CELL 2 — Imports + Google Drive mount

import os
import re
import json
import random
import asyncio
import traceback
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import quote, unquote, urlparse, parse_qsl, urlencode, urlunparse, urljoin

import pandas as pd
import nest_asyncio
import gspread

from google.colab import auth, drive
from google.auth import default

from playwright.async_api import async_playwright

nest_asyncio.apply()

# Mount Drive if needed.
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("Google Drive is already mounted.")

print("Environment loaded.")


In [ ]:

# CELL 3 — Google Sheets connection

GOOGLE_SHEET_NAME = (
    "Restaurant_Profiling_V2_2.1_data structure GOOGLE SHEET"
)

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

try:
    spreadsheet = gc.open(
        GOOGLE_SHEET_NAME
    )

    print(
        f"Connected to Google Sheet: "
        f"{GOOGLE_SHEET_NAME}"
    )

    print("Worksheets:")
    for ws in spreadsheet.worksheets():
        print(
            f" - {ws.title} "
            f"({ws.id})"
        )

except Exception as exc:
    raise RuntimeError(
        "Could not open the configured Google Sheet. "
        "Check GOOGLE_SHEET_NAME and your Google account permissions."
    ) from exc


In [ ]:

# CELL 4 — Configuration

# ==========================================================
# RUN MODES
# ==========================================================

RUN_REFRESH = True
RUN_SEARCH = True

# ==========================================================
# SAFE FIRST-TEST LIMITS
# ==========================================================

MAX_REFRESH_RECORDS = 10
MAX_SEARCH_QUERIES = 2
MAX_RESULTS_PER_QUERY = 10

# FULL RUN:
# MAX_REFRESH_RECORDS = None
# MAX_SEARCH_QUERIES = None
# MAX_RESULTS_PER_QUERY = 18

# ==========================================================
# SEARCH
# ==========================================================

TARGET_PRIORITY = "P1"

SEARCH_CATEGORIES = [
    "restaurants",
    "cafes",
    "fast food",
    "Pakistani restaurants",
    "BBQ restaurants",
    "Chinese restaurants",
    "pizza restaurants",
    "burger restaurants",
    "coffee shops",
    "bakery",
    "dessert restaurants",
    "fine dining restaurants",
]

# ==========================================================
# BROWSER
# ==========================================================

HEADLESS = True
VIEWPORT = {
    "width": 1440,
    "height": 900,
}

MIN_DELAY = 1.5
MAX_DELAY = 3.5

PAGE_WAIT_MIN = 1.5
PAGE_WAIT_MAX = 3.0

MAX_NAV_RETRIES = 3
RETRY_BASE_DELAY = 5
CAPTCHA_PAUSE = 60

# ==========================================================
# GOOGLE DRIVE OUTPUTS
# ==========================================================

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "Analytics as a Service/BI Services/"
    "Rest Anchor/Scrapper Output"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

CHECKPOINT_JSON = os.path.join(
    OUTPUT_DIR,
    "v2_3_2_checkpoint.json"
)

SCRAPE_LOG_CSV = os.path.join(
    OUTPUT_DIR,
    "v2_3_2_scrape_log.csv"
)

RAW_RESULTS_JSON = os.path.join(
    OUTPUT_DIR,
    "v2_3_2_raw_results.json"
)

RESET_CHECKPOINT = False

print("Configuration loaded.")
print("Drive output:", OUTPUT_DIR)

# A run resumes only its own unfinished work. Change this for a fresh collection.
RUN_ID = datetime.now(timezone.utc).strftime("%Y-%m-%d")
CHECKPOINT_MAX_AGE_HOURS = 24
REPAIR_DASHBOARD = False

# Wait for actual result links; an empty feed is not a completed search.
SEARCH_READY_TIMEOUT_MS = 45_000
MAX_SEARCH_SCROLLS = 20
SEARCH_SCROLL_WAIT_MS = 2_000


In [ ]:

# CELL 5 — Google Sheet validation + robust table reader

REQUIRED_SHEETS = [
    "Restaurant Master",
    "ISB-RWP Target Areas",
]

MASTER_COLUMNS = [
    "Restaurant ID",
    "Restaurant Name",
    "Brand Name",
    "Branch #",
    "Format",
    "Cuisine Types",
    "City",
    "Area",
    "Full Address",
    "Latitude",
    "Longitude",
    "Operational Timing",
    "Phone Number",
    "Owner/Manager Contact",
    "Owner/Manager Name",
    "LinkedIn (Y/N)",
    "Dine-In (Y/N)",
    "Takeaway (Y/N)",
    "Delivery (Y/N)",
    "POS Installed (Y/N)",
    "POS System Name",
    "Dedicated BI Team (Y/N)",
    "Website URL",
    "Website Type",
    "Foodpanda Listed (Y/N)",
    "Foodpanda URL",
    "Own Delivery Network (Y/N)",
    "Delivery Partner(s)",
    "Google Business (Y/N)",
    "Google Maps Link",
    "Google Star Rating",
    "Google Review Count",
    "Avg Ticket Size (PKR)",
    "Avg Daily Orders",
    "Data Source",
    "Date Profiled",
    "Status",
    "Notes",
]

V231_COLUMNS = [
    "Place ID",
    "Google Category",
    "Current Open Status",
    "Business Status",
    "Price Level",
    "Data Confidence",
    "Instagram URL",
    "Facebook URL",
    "Last Observed At",
]


def normalize_text(value):
    if value is None:
        return ""
    return re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()


def clean_sheet_values(values):
    """
    Convert get_all_values() output into a DataFrame.
    Keeps row positions aligned with actual Google Sheet rows.
    """
    if not values:
        return pd.DataFrame()

    width = max(
        len(row)
        for row in values
    )

    padded = [
        list(row) + [""] * (
            width - len(row)
        )
        for row in values
    ]

    headers = [
        normalize_text(x)
        for x in padded[0]
    ]

    # If duplicate/blank headers exist, surface a useful warning.
    duplicates = [
        h
        for h in set(headers)
        if h and headers.count(h) > 1
    ]

    if duplicates:
        raise ValueError(
            "Duplicate Google Sheet headers found: "
            + ", ".join(duplicates)
        )

    df = pd.DataFrame(
        padded[1:],
        columns=headers
    )

    # Remove entirely blank data rows.
    if not df.empty:
        df = df.loc[
            ~df.apply(
                lambda row: all(
                    normalize_text(v) == ""
                    for v in row
                ),
                axis=1
            )
        ].reset_index(
            drop=True
        )

    return df


def get_sheet_dataframe(worksheet):
    values = worksheet.get_all_values()
    return clean_sheet_values(values)


def validate_google_sheet():

    titles = [
        ws.title
        for ws in spreadsheet.worksheets()
    ]

    missing = [
        s
        for s in REQUIRED_SHEETS
        if s not in titles
    ]

    if missing:
        raise ValueError(
            "Missing required sheets: "
            + ", ".join(missing)
        )

    master_ws = spreadsheet.worksheet(
        "Restaurant Master"
    )

    master_values = master_ws.get_all_values()

    if not master_values:
        raise ValueError(
            "Restaurant Master is empty."
        )

    headers = [
        normalize_text(x)
        for x in master_values[0]
    ]

    missing_columns = [
        c
        for c in MASTER_COLUMNS
        if c not in headers
    ]

    if missing_columns:
        raise ValueError(
            "Restaurant Master is missing columns: "
            + ", ".join(missing_columns)
        )

    print(
        "Google Sheet validation passed."
    )
    print(
        "Restaurant Master data rows:",
        max(0, len(master_values) - 1)
    )

    print(
        "Restaurant Master columns:",
        len(headers)
    )

    print("\nColumns:")
    for i, h in enumerate(
        headers,
        start=1
    ):
        print(
            f"{i:02d}. {h}"
        )

    return master_ws


master_ws = validate_google_sheet()


In [ ]:

# CELL 6 — Parsing helpers

def normalize_name(value):
    # Preserve Urdu and other Unicode names.
    return re.sub(r"[^\w]+", " ", normalize_text(value).casefold(), flags=re.UNICODE).strip()


def normalize_phone(value):
    value = normalize_text(value)

    if not value:
        return ""

    return re.sub(
        r"[^\d+\-\(\)\s]",
        "",
        value
    ).strip()


def phone_key(value):
    return re.sub(
        r"\D",
        "",
        normalize_phone(value)
    )


def first_nonempty(*values):
    for value in values:
        if value not in (
            None,
            "",
            [],
            {}
        ):
            return value
    return None


def parse_count(value):
    if value is None:
        return None

    value = (
        normalize_text(value)
        .upper()
        .replace(",", "")
    )

    if not value:
        return None

    for suffix, multiplier in [
        ("K", 1_000),
        ("M", 1_000_000),
        ("B", 1_000_000_000),
    ]:
        if value.endswith(suffix):
            try:
                return int(
                    float(
                        value[:-1]
                    ) * multiplier
                )
            except Exception:
                return None

    if value.isdigit():
        return int(value)

    return None


def parse_rating(value):
    text = normalize_text(value)
    match = re.fullmatch(r"([0-5](?:\.\d+)?)", text)
    if not match:
        match = re.search(r"(?<![\d.])([0-5](?:\.\d+)?)\s*(?:stars?\b|/\s*5\b)", text, re.I)
    if match and 0 <= float(match.group(1)) <= 5:
        return float(match.group(1))
    return None


def extract_coordinates(url):
    # !3d/!4d identifies the place; @ identifies the camera viewport, not the pin.
    match = re.search(r"!3d(-?\d+(?:\.\d+)?)!4d(-?\d+(?:\.\d+)?)", unquote(url or ""))
    if match:
        lat, lon = map(float, match.groups())
        if -90 <= lat <= 90 and -180 <= lon <= 180:
            return lat, lon
    return None, None


def parse_day_name(text):

    mapping = {
        "mon": "Mon",
        "monday": "Mon",
        "tue": "Tue",
        "tues": "Tue",
        "tuesday": "Tue",
        "wed": "Wed",
        "wednesday": "Wed",
        "thu": "Thu",
        "thur": "Thu",
        "thurs": "Thu",
        "thursday": "Thu",
        "fri": "Fri",
        "friday": "Fri",
        "sat": "Sat",
        "saturday": "Sat",
        "sun": "Sun",
        "sunday": "Sun",
    }

    return mapping.get(
        normalize_text(
            text
        ).lower()
    )

print("Parsing helpers loaded.")


In [ ]:
def normalize_maps_url(url):
    value = normalize_text(url)
    if not value:
        return ""
    place_id = extract_place_id_from_url(value)
    if place_id:
        return "place_id:" + place_id
    internal = extract_maps_internal_id(value)
    if internal:
        return "maps_internal:" + internal.lower()
    parts = urlparse(value)
    params = sorted((k, v) for k, v in parse_qsl(parts.query) if k not in {"hl", "authuser", "entry", "g_ep"} and not k.startswith("utm_"))
    return urlunparse((parts.scheme, parts.netloc.lower(), parts.path.rstrip("/"), "", urlencode(params), ""))


# CELL 7 — Identity + classification helpers

def extract_place_id_from_url(url):
    """
    Extract genuine ChIJ-style Place IDs.
    Do NOT treat 0x...:0x... grid/internal IDs as Place IDs.
    """
    if not url:
        return ""

    decoded = unquote(url)

    patterns = [
        r"!1s(ChIJ[A-Za-z0-9_\-]+)",
        r"!19s(ChIJ[A-Za-z0-9_\-]+)",
        r"[?&]query_place_id=(ChIJ[A-Za-z0-9_\-]+)",
        r"place_id=(ChIJ[A-Za-z0-9_\-]+)",
    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            decoded
        )

        if match:
            return match.group(1)

    return ""


def extract_maps_internal_id(url):

    if not url:
        return ""

    decoded = unquote(
        url
    )

    match = re.search(
        r"!1s(0x[0-9a-fA-F]+:0x[0-9a-fA-F]+)",
        decoded
    )

    if match:
        return match.group(1)

    return ""


def make_identity_key(record):

    place_id = normalize_text(
        record.get("place_id")
    ) or extract_place_id_from_url(record.get("maps_url", ""))

    if place_id:
        return (
            "place_id:"
            + place_id
        )

    maps_url = normalize_text(
        record.get("maps_url")
    )

    if maps_url:
        return (
            "maps:"
            + normalize_maps_url(maps_url)
        )

    phone = phone_key(
        record.get("phone")
    )

    if phone:
        return (
            "phone:"
            + phone
        )

    name = normalize_name(
        record.get("name")
    )

    address = normalize_name(
        record.get("address")
    )

    if name and address:
        return (
            f"name_address:{name}|{address}"
        )

    lat = record.get("latitude")
    lon = record.get("longitude")

    if (
        name
        and lat is not None
        and lon is not None
    ):
        return (
            f"name_coord:{name}|"
            f"{round(float(lat), 5)}|"
            f"{round(float(lon), 5)}"
        )

    return (
        "name:"
        + name
    )


def infer_format(
    category,
    name
):

    cat = normalize_text(
        category
    ).lower()

    name_text = normalize_text(
        name
    ).lower()

    if cat:

        if any(
            x in cat
            for x in [
                "coffee",
                "cafe",
                "café"
            ]
        ):
            return "Cafe"

        if any(
            x in cat
            for x in [
                "bakery",
                "bakers"
            ]
        ):
            return "Bakery"

        if any(
            x in cat
            for x in [
                "ice cream",
                "dessert",
                "sweets",
                "mithai"
            ]
        ):
            return "Dessert / Sweets"

        if any(
            x in cat
            for x in [
                "fast food",
                "burger",
                "pizza",
                "qsr",
                "fried chicken"
            ]
        ):
            return "QSR"

        if any(
            x in cat
            for x in [
                "fine dining",
                "steakhouse",
                "steak house"
            ]
        ):
            return "Full-Service Restaurant"

        if any(
            x in cat
            for x in [
                "dhaba",
                "dhabba"
            ]
        ):
            return "Dhaba"

        if "food court" in cat or "food truck" in cat:
            return "Food Court / Truck"

        if "restaurant" in cat:
            return "Restaurant"

    if (
        "coffee" in name_text
        or "cafe" in name_text
        or "café" in name_text
    ):
        return "Cafe"

    if (
        "bakery" in name_text
        or "bakers" in name_text
    ):
        return "Bakery"

    if any(
        x in name_text
        for x in [
            "burger",
            "pizza",
            "fried chicken"
        ]
    ):
        return "QSR"

    return "Restaurant"


KNOWN_BRANDS = [
    "Monal",
    "Tuscany Courtyard",
    "Burning Brownie",
    "JEETO",
    "Savour Foods",
    "Jalal Sons",
    "Tehzeeb Bakers",
    "Gourmet",
    "Student Biryani",
    "Bundu Khan",
    "Bar.B.Q Tonight",
    "BBQ Tonight",
    "Howdy",
    "KFC",
    "McDonald's",
    "Pizza Hut",
    "Domino's",
    "Hardee's",
    "Subway",
    "Nando's",
    "OPTP",
    "Broadway Pizza",
    "14th Street Pizza",
    "Kababjees",
    "Kolachi",
    "Café Aylanto",
    "Chaaye Khana",
    "Mocca Coffee",
    "Gloria Jean's",
    "Dunkin' Donuts",
    "Baskin Robbins",
    "Cold Stone",
    "Layers Bakeshop",
    "Pie in the Sky",
    "Roasters",
    "Juice Junction",
    "Des Pardes",
    "Haveli",
    "Second Cup",
    "Zus Coffee",
    "KAF Coffee",
    "Eggspectation",
    "Caffé Praha",
]


def conservative_brand(name):

    name = normalize_text(
        name
    )

    if not name:
        return ""

    lower = name.lower()

    for brand in KNOWN_BRANDS:

        if brand.lower() in lower:
            return brand

    # IMPORTANT:
    # Split only on spaced separators so F-6 is not split.
    value = re.split(
        r"\s+-\s+|\s*\|\s*|\s*,\s+",
        name
    )[0].strip()

    value = re.sub(
        r"\b(DHA|Blue Area|Saddar|Bahria Town|"
        r"Johar Town|Gulberg)\b.*$",
        "",
        value,
        flags=re.IGNORECASE
    ).strip(
        " -|,"
    )

    return value


def classify_website_type(url):

    if not url:
        return ""

    lower = url.lower()

    mapping = [
        ("foodpanda", "Foodpanda"),
        ("eat.cheetay", "Cheetay"),
        ("careem.com", "Careem"),
        ("bykea.com", "Bykea"),
        ("instagram.com", "Instagram"),
        ("facebook.com", "Facebook"),
        ("fb.com", "Facebook"),
        ("tiktok.com", "TikTok"),
        ("linktree", "Linktree"),
        ("linktr.ee", "Linktree"),
    ]

    for key, label in mapping:

        if key in lower:
            return label

    return "Own Website"


def infer_confidence(record):

    problems = []

    if not record.get("name"):
        problems.append(
            "name missing"
        )

    if not record.get("maps_url"):
        problems.append(
            "Maps URL missing"
        )

    if not record.get("place_id"):
        problems.append(
            "Place ID missing"
        )

    if record.get("rating") is None:
        problems.append(
            "rating missing"
        )

    if record.get("review_count") is None:
        problems.append(
            "review count missing"
        )

    if not record.get("address"):
        problems.append(
            "address missing"
        )

    if not record.get("phone"):
        problems.append(
            "phone missing"
        )

    if record.get(
        "review_source"
    ) == "body-fallback":
        problems.append(
            "review count from fallback"
        )

    for field in ("latitude", "longitude", "category"):
        if record.get(field) in (None, ""):
            problems.append(field + " missing")

    if record.get("error"):
        problems.append(
            "extraction error"
        )

    if problems:
        return (
            "REVIEW: "
            + "; ".join(problems)
        )

    return "OK"

print("Identity/classification helpers loaded.")


In [ ]:
# CELL 8 — Browser/navigation helpers

class MapsAccessError(RuntimeError):
    """Google returned a consent, access, or browser error page."""

class SearchCollectionError(RuntimeError):
    """Discovery did not produce a verified results/empty state."""


def maps_search_url(query):
    # Official Maps URL syntax; does not require a Places API key.
    return "https://www.google.com/maps/search/?" + urlencode({"api": "1", "query": query, "hl": "en"})


def listing_url(href, base_url="https://www.google.com/maps/"):
    """Accept only URLs identifying an individual Google Maps listing."""
    url = urljoin(base_url, normalize_text(href))
    parsed = urlparse(url)
    if parsed.scheme not in {"http", "https"} or parsed.hostname not in {"google.com", "www.google.com", "maps.google.com", "maps.app.goo.gl", "goo.gl"}:
        return ""
    params = dict(parse_qsl(parsed.query))
    if parsed.hostname == "maps.app.goo.gl" or (parsed.hostname == "goo.gl" and parsed.path.startswith("/maps/")):
        return url
    is_maps_path = parsed.path.startswith("/maps") or (parsed.hostname == "maps.google.com" and parsed.path in {"", "/"})
    if parsed.path.startswith("/maps/place/") or (is_maps_path and (params.get("query_place_id") or params.get("cid") or params.get("ftid") or params.get("q", "").startswith("place_id:"))):
        return url
    return ""


def classify_maps_page(snapshot):
    text = normalize_text(snapshot.get("body", "")).casefold()
    url = snapshot.get("url", "").casefold()
    status = snapshot.get("http_status")
    if status in {401, 403, 429} or "/sorry/" in url or any(x in text for x in ["unusual traffic", "automated queries", "verify you are human", "verify that you're not a robot", "are you a robot", "access denied"]):
        return "blocked"
    if "consent.google." in url or "before you continue to google" in text:
        return "consent"
    if url.startswith("chrome-error:") or any(x in text for x in ["err_name_not_resolved", "err_connection", "err_proxy_connection", "this site can’t be reached", "this site can't be reached"]):
        return "network_error"
    if isinstance(status, int) and status >= 400:
        return "http_error"
    if snapshot.get("detail_name") and listing_url(snapshot.get("url", "")):
        return "single_place"
    if snapshot.get("listing_count", 0):
        return "results"
    if any(re.fullmatch(r"(?:No results found(?: for .+)?|No results|We (?:couldn't|could not) find .+|Google Maps (?:can't|cannot) find .+)[.!]?", normalize_text(line), re.I) for line in snapshot.get("body", "").splitlines()):
        return "no_results"
    if "enable javascript" in text or "javascript is disabled" in text:
        return "javascript_required"
    return "unrecognized"


async def discover_listing_links(page):
    # Google can render links as relative URLs, CID links, or place-ID controls.
    raw = await page.locator('a[href], [role="feed"] [data-place-id], [role="feed"] [data-cid]').evaluate_all(
        """els => els.filter(e => !e.closest('[data-review-id]')).map(e => ({
            href: e.getAttribute('href'), place_id: e.getAttribute('data-place-id'),
            cid: e.getAttribute('data-cid'), label: e.getAttribute('aria-label') || e.innerText || ''
        }))""")
    results = {}
    for item in raw:
        href = item.get("href") or ""
        if not href and item.get("place_id"):
            href = "https://www.google.com/maps/search/?" + urlencode({"api": "1", "query": normalize_text(item.get("label")) or "restaurant", "query_place_id": item["place_id"]})
        if not href and str(item.get("cid") or "").isdigit():
            href = "https://www.google.com/maps?cid=" + str(item["cid"])
        url = listing_url(href, page.url) if href else ""
        if url:
            results[normalize_maps_url(url)] = {"url": url, "label": normalize_text(item.get("label"))}
    return list(results.values())


async def inspect_maps_page(page):
    snapshot = {"url": page.url, "http_status": getattr(page, "_collector_http_status", None)}
    errors = []
    try:
        snapshot["title"] = await page.title()
        snapshot["body"] = (await page.locator("body").inner_text(timeout=5000))[:12000]
        snapshot["feed_count"] = await page.locator('[role="feed"]').count()
        snapshot["anchor_count"] = await page.locator('a[href]').count()
        snapshot["listing_count"] = len(await discover_listing_links(page))
        h1 = place_panel(page).locator("h1").first
        snapshot["detail_name"] = normalize_text(await h1.inner_text(timeout=1500)) if await h1.count() else ""
        if snapshot["detail_name"].casefold() in {"results", "search results", "google maps"}:
            snapshot["detail_name"] = ""
    except Exception as exc:
        errors.append(f"{type(exc).__name__}: {str(exc)[:400]}")
    snapshot["inspection_errors"] = errors
    snapshot["network_events"] = getattr(page, "_collector_network_events", [])[-30:]
    snapshot["state"] = classify_maps_page(snapshot)
    return snapshot


async def save_search_debug(page, query, reason, snapshot=None):
    directory = Path(OUTPUT_DIR) / "search_debug"
    directory.mkdir(parents=True, exist_ok=True)
    stem = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f") + "_" + re.sub(r"[^a-zA-Z0-9]+", "_", query)[:60]
    snapshot = snapshot or await inspect_maps_page(page)
    report = {"query": query, "reason": reason, "captured_at": now_iso(), **snapshot, "files": {}}
    for kind, suffix in [("screenshot", ".png"), ("html", ".html")]:
        path = directory / (stem + suffix)
        try:
            if kind == "screenshot":
                await page.screenshot(path=str(path), full_page=False, timeout=10000)
            else:
                path.write_text(await page.content(), encoding="utf-8")
            report["files"][kind] = str(path)
        except Exception as exc:
            report["files"][kind + "_error"] = str(exc)[:300]
    report_path = directory / (stem + ".json")
    report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    print(f"  SEARCH DIAGNOSTIC: {snapshot.get('state')} | HTTP={snapshot.get('http_status')} | feeds={snapshot.get('feed_count', 0)} | listing links={snapshot.get('listing_count', 0)}")
    print("  Page title:", snapshot.get("title", ""))
    print("  Final URL:", snapshot.get("url", ""))
    print("  Page text:", normalize_text(snapshot.get("body", ""))[:600])
    print("  Debug report:", report_path)
    if report["files"].get("screenshot"):
        print("  Screenshot:", report["files"]["screenshot"])
        try:
            from IPython.display import Image, display
            display(Image(filename=report["files"]["screenshot"], width=900))
        except ImportError:
            pass
    return str(report_path)


async def accept_google_consent(page):
    # Reject optional cookies where the dialog provides that option.
    for frame in page.frames:
        for label in ["Reject all", "Accept all", "I agree", "Accept"]:
            try:
                button = frame.get_by_role("button", name=re.compile(rf"^{re.escape(label)}$", re.I)).first
                if await button.count() and await button.is_visible():
                    await button.click(timeout=2500)
                    await page.wait_for_timeout(1000)
                    return True
            except Exception:
                continue
    return False


def attach_navigation_diagnostics(page):
    if hasattr(page, "_collector_network_events"):
        page._collector_network_events.clear()
        return
    page._collector_network_events = []
    def remember(kind, value):
        page._collector_network_events.append({"kind": kind, "message": str(value)[:400]})
        del page._collector_network_events[:-30]
    page.on("pageerror", lambda error: remember("javascript", error))
    page.on("requestfailed", lambda request: remember("request_failed", f"{request.resource_type}: {request.failure} | {urlparse(request.url).hostname}"))


async def detect_captcha(page):
    body = await page.locator("body").inner_text(timeout=5000)
    return classify_maps_page({"url": page.url, "body": body, "http_status": getattr(page, "_collector_http_status", None)}) == "blocked"


async def safe_goto(page, url, wait_ms=None):
    attach_navigation_diagnostics(page)
    last_error = None
    for attempt in range(MAX_NAV_RETRIES):
        try:
            page._collector_http_status = None
            response = await page.goto(url, wait_until="domcontentloaded", timeout=60000)
            page._collector_http_status = response.status if response else None
            await accept_google_consent(page)
            if await detect_captcha(page):
                raise MapsAccessError("Google blocked this session or requested human verification. Stop this run; no automatic bypass is attempted.")
            await page.wait_for_timeout(wait_ms if wait_ms is not None else random.randint(int(PAGE_WAIT_MIN * 1000), int(PAGE_WAIT_MAX * 1000)))
            return
        except MapsAccessError:
            raise
        except Exception as exc:
            last_error = exc
            if attempt + 1 < MAX_NAV_RETRIES:
                delay = min(RETRY_BASE_DELAY * 2 ** attempt, 30)
                print(f"  Navigation retry {attempt + 1}/{MAX_NAV_RETRIES}: {type(exc).__name__}; waiting {delay}s")
                await asyncio.sleep(delay)
    raise last_error or RuntimeError("Navigation failed")


def place_panel(page):
    return page.locator('[role="main"]').filter(has=page.locator("h1")).first


async def structured_values(root, selector, attributes=("aria-label", "data-tooltip"), include_text=False):
    return await root.locator(selector).evaluate_all(
        """(els, config) => els.filter(e => !e.closest('[role="feed"], [data-review-id]'))
        .flatMap(e => [...config.attrs.map(a => e.getAttribute(a)), config.text ? e.innerText : null])
        .filter(Boolean)""", {"attrs": list(attributes), "text": include_text})

print("Browser helpers and search diagnostics loaded (V2.3.3).")


In [ ]:
# CELL 9 — Review + rating extraction, scoped to listing summary controls

async def extract_review_count(page):
    root = place_panel(page)
    selector = 'button[aria-label*="review" i], button[aria-label*="rating" i], [role="img"][aria-label*="star" i], [jsaction*="moreReviews"], button[jsaction*="pane.rating"]'
    try:
        labels = await structured_values(root, selector)
        texts = await structured_values(root, selector, attributes=(), include_text=True)
        for source, candidates in [("aria-label", labels), ("review-control", texts)]:
            for raw in candidates:
                text = normalize_text(raw)
                match = re.search(r"(?<![\d.])([\d,]+(?:\.\d+)?[KMB]?)\s+(?:reviews?|ratings?)\b", text, re.I)
                if not match and source == "review-control":
                    match = re.fullmatch(r"\(?([\d,]+(?:\.\d+)?[KMB]?)\)?", text, re.I)
                if match:
                    count = parse_count(match.group(1))
                    if count is not None:
                        return {"value": count, "source": source, "raw": text}
    except Exception:
        pass
    # Do not read review text or unrelated listing cards as a last resort.
    return {"value": None, "source": "not-found", "raw": ""}

async def extract_rating(page):
    try:
        values = await structured_values(place_panel(page), '[role="img"][aria-label*="star" i], [jsaction*="moreReviews"]')
        for value in values:
            rating = parse_rating(value)
            if rating is not None:
                return rating
    except Exception:
        pass
    return None


In [ ]:

# CELL 10 — Operating hours extraction

TIME_TOKEN_RE = re.compile(
    r"\b(?:1[0-2]|0?[1-9])"
    r"(?::[0-5]\d)?\s*"
    r"(?:AM|PM|am|pm)"
)

VALID_SPECIAL_HOURS = {
    "closed",
    "open 24 hours",
    "24 hours",
    "open all day",
}


def extract_time_range(text):
    text = normalize_text(text).replace("a.m.", "AM").replace("p.m.", "PM")
    if text.lower() in VALID_SPECIAL_HOURS:
        return "Closed" if text.lower() == "closed" else "Open 24 hours"
    # Full match rejects UI garbage and incomplete/single times. Retain split shifts.
    token = r"(?:1[0-2]|0?[1-9])(?::[0-5]\d)?"
    interval = rf"({token})\s*(AM|PM)?\s*(?:to|–|-|—)\s*({token})\s*(AM|PM)"
    parts = re.split(r"\s*[,;]\s*", text)
    result = []
    for part in parts:
        match = re.fullmatch(interval, part, re.I)
        if not match:
            return ""
        start, start_period, end, end_period = match.groups()
        result.append(f"{start} {(start_period or end_period).upper()}–{end} {end_period.upper()}")
    return ", ".join(result)


def is_valid_hour_value(value):
    return bool(extract_time_range(value))


async def extract_hours(page):
    root = place_panel(page)
    found = {}
    # The weekly hours table is commonly collapsed behind the hours control.
    try:
        button = root.locator('[data-item-id="oh"], button[aria-label*="hours" i]').first
        if await button.count():
            await button.click(timeout=2000)
            await root.locator("table tr").first.wait_for(state="visible", timeout=2000)
    except Exception:
        pass
    try:
        labels = await structured_values(root, '[aria-label], [data-tooltip]')
        rows = await root.locator("table tr").evaluate_all("els => els.map(e => e.innerText)")
        for raw in labels + rows:
            text = normalize_text(raw)
            match = re.fullmatch(r"(Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)\s*[:,]?\s+(.+)", text, re.I)
            if match:
                value = extract_time_range(match.group(2))
                if value:
                    found[parse_day_name(match.group(1))] = value
    except Exception:
        pass
    return " | ".join(f"{d}: {found[d]}" for d in ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] if d in found)


def compress_hours(hours):

    if not hours:
        return ""

    parsed = []

    for part in hours.split("|"):

        part = normalize_text(
            part
        )

        match = re.match(
            r"^(Mon|Tue|Wed|Thu|Fri|Sat|Sun)"
            r"\s*:\s*(.+)$",
            part
        )

        if not match:
            continue

        day = match.group(1)
        value = normalize_text(
            match.group(2)
        )

        if not is_valid_hour_value(
            value
        ):
            continue

        parsed.append(
            (day, value)
        )

    if not parsed:
        return ""

    if len(parsed) < 7:

        return " | ".join(
            f"{d}: {v}"
            for d, v in parsed
        )

    groups = []

    start_day = parsed[0][0]
    current_value = parsed[0][1]

    for i in range(
        1,
        len(parsed)
    ):

        if parsed[i][1] == current_value:
            continue

        groups.append(
            (
                start_day,
                parsed[i - 1][0],
                current_value
            )
        )

        start_day = parsed[i][0]
        current_value = parsed[i][1]

    groups.append(
        (
            start_day,
            parsed[-1][0],
            current_value
        )
    )

    parts = []

    for start, end, value in groups:

        if start == end:
            parts.append(
                f"{start}: {value}"
            )
        else:
            parts.append(
                f"{start}–{end}: {value}"
            )

    return " | ".join(
        parts
    )

print("Operating-hours extractor loaded.")

def is_valid_schedule(hours):
    parts = normalize_text(hours).split("|")
    for part in parts:
        match = re.fullmatch(r"(Mon|Tue|Wed|Thu|Fri|Sat|Sun)(?:[–-](Mon|Tue|Wed|Thu|Fri|Sat|Sun))?\s*:\s*(.+)", part.strip())
        if not match or not is_valid_hour_value(match.group(3)):
            return False
    return bool(parts)


In [ ]:

# CELL 11 — Service signals + status

def detect_service_token(text):
    result = {"dine_in": "", "takeaway": "", "delivery": ""}
    # Accept only explicit service labels, never arbitrary sentences mentioning a service.
    aliases = {"dine_in": r"dine[ -]?in", "takeaway": r"take[ -]?(?:away|out)", "delivery": r"(?:no-contact |contactless )?delivery"}
    text = re.sub(r"^service options?\s*:\s*", "", normalize_text(text), flags=re.I)
    for part in re.split(r"[·•,;|]", text):
        part = part.strip()
        for key, pattern in aliases.items():
            if re.fullmatch(rf"(?:offers? |has |✓\s*)?{pattern}(?: available)?", part, re.I):
                result[key] = "Y"
    return result


async def extract_service_signals(page):
    result = {"dine_in": "", "takeaway": "", "delivery": ""}
    try:
        candidates = await structured_values(place_panel(page), '[aria-label], [data-tooltip], button[data-item-id*="service"]', include_text=False)
        for candidate in candidates:
            for key, value in detect_service_token(candidate).items():
                if value == "Y":
                    result[key] = value
    except Exception:
        pass
    return result


async def extract_business_status(page):
    result = {"current_open_status": "Unknown", "business_status": "Unknown"}
    try:
        values = await structured_values(place_panel(page), '[data-item-id="oh"], [role="status"], [aria-label]', include_text=True)
        for raw in values:
            text = normalize_text(raw)
            if re.fullmatch(r"(?:This (?:place|business) is )?permanently closed[.]?", text, re.I):
                return {"current_open_status": "Closed", "business_status": "Permanently Closed"}
            if re.fullmatch(r"(?:This (?:place|business) is )?temporarily closed[.]?", text, re.I):
                result = {"current_open_status": "Closed", "business_status": "Temporarily Closed"}
            elif result["business_status"] != "Temporarily Closed":
                if re.match(r"^Open(?: now| 24 hours)?(?:$|\s*[·⋅])", text, re.I):
                    result = {"current_open_status": "Open", "business_status": "Operational"}
                elif re.match(r"^Closed(?: now)?(?:$|\s*[·⋅])", text, re.I):
                    result = {"current_open_status": "Closed", "business_status": "Operational"}
    except Exception:
        pass
    return result


async def extract_price_level(page):

    try:

        values = await place_panel(page).locator(
            "[aria-label]"
        ).evaluate_all(
            """
            els => els
              .map(e => e.getAttribute('aria-label'))
              .filter(Boolean)
              .filter(v => /price|price range|\\$\\$|moderately|inexpensive|expensive/i.test(v))
              .slice(0, 100)
            """
        )

        for value in values:

            match = re.search(
                r"(\${1,5})",
                value
            )

            if match:
                return match.group(1)

    except Exception:
        pass

    return ""



print("Service/status/social extractors loaded.")


In [ ]:

# CELL 12 — Core business extractors

async def extract_phone(page):

    selectors = [
        'button[data-item-id^="phone:tel:"]',
        'button[aria-label*="Phone" i]',
        'a[href^="tel:"]',
    ]

    for selector in selectors:

        try:

            locator = place_panel(page).locator(
                selector
            ).first

            if await locator.count():

                aria = await locator.get_attribute(
                    "aria-label"
                )

                text = await locator.inner_text()

                value = first_nonempty(
                    aria,
                    text
                )

                if value:

                    value = re.sub(
                        r"^Phone\s*:\s*",
                        "",
                        value,
                        flags=re.IGNORECASE
                    )

                    value = normalize_phone(
                        value
                    )

                    if value:
                        return value

        except Exception:
            pass

    return ""


async def extract_address(page):

    selectors = [
        'button[data-item-id="address"]',
        'button[aria-label^="Address:" i]',
        'button[aria-label*="Address" i]',
    ]

    for selector in selectors:

        try:

            locator = place_panel(page).locator(
                selector
            ).first

            if await locator.count():

                aria = await locator.get_attribute(
                    "aria-label"
                )

                text = await locator.inner_text()

                value = first_nonempty(
                    aria,
                    text
                )

                if value:

                    value = re.sub(
                        r"^Address\s*:\s*",
                        "",
                        value,
                        flags=re.IGNORECASE
                    )

                    return normalize_text(
                        value
                    )

        except Exception:
            pass

    return ""


async def extract_website(page):

    selectors = [
        'a[data-item-id="authority"]',
        'a[aria-label*="Website" i]',
    ]

    for selector in selectors:

        try:

            locator = place_panel(page).locator(
                selector
            ).first

            if await locator.count():

                href = await locator.get_attribute(
                    "href"
                )

                if href:
                    return href

        except Exception:
            pass

    return ""


async def extract_google_category(
    page,
    name
):

    selectors = [
        'button[jsaction*="category"]',
    ]

    for selector in selectors:

        try:

            elements = place_panel(page).locator(
                selector
            )

            count = await elements.count()

            for i in range(
                min(count, 20)
            ):

                try:

                    text = normalize_text(
                        await elements.nth(i).inner_text()
                    )

                    if (
                        text
                        and text.lower()
                        != normalize_text(
                            name
                        ).lower()
                        and 2 <= len(text) <= 100
                    ):
                        return text

                except Exception:
                    pass

        except Exception:
            pass

    return ""

print("Core extractors loaded.")


In [ ]:

# CELL 13 — SINGLE shared place-detail extractor

async def extract_place_details(
    page,
    *,
    url,
    city="",
    area="",
    search_category="",
    mode="SEARCH"
):

    record = {
        "mode": mode,
        "observed_at": now_iso(),
        "requested_url": url,
        "city": city,
        "area": area,
        "search_category": search_category,

        "name": "",
        "category": "",
        "address": "",
        "phone": "",
        "website": "",
        "website_type": "",

        "rating": None,
        "review_count": None,
        "review_source": "not-found",
        "review_raw": "",

        "hours": "",
        "latitude": None,
        "longitude": None,

        "dine_in": "",
        "takeaway": "",
        "delivery": "",

        "current_open_status": "Unknown",
        "business_status": "Unknown",
        "price_level": "",

        "maps_url": "",
        "place_id": "",
        "maps_internal_id": "",

        "format": "",
        "rough_brand": "",

        "data_confidence": "",
        "error": "",

        "instagram_url": "",
        "facebook_url": "",
        "foodpanda_url": "",
        "foodpanda_listed": "",
    }

    try:

        await safe_goto(
            page,
            url
        )

        record["maps_url"] = page.url

        # Name
        try:

            h1 = place_panel(page).locator(
                "h1"
            ).first

            if await h1.count():

                record["name"] = normalize_text(
                    await h1.inner_text()
                )

        except Exception:
            pass

        if not record["name"] or record["name"].casefold() in {"results", "google maps", "search results"}:

            record["error"] = (
                "Restaurant name not found"
            )

            record["data_confidence"] = (
                "REVIEW: name missing"
            )

            return record

        # Core
        record["rating"] = (
            await extract_rating(page)
        )

        review = await extract_review_count(
            page
        )

        record["review_count"] = (
            review["value"]
        )

        record["review_source"] = (
            review["source"]
        )

        record["review_raw"] = (
            review["raw"]
        )

        record["address"] = (
            await extract_address(page)
        )

        record["phone"] = (
            await extract_phone(page)
        )

        record["website"] = (
            await extract_website(page)
        )

        record["website_type"] = (
            classify_website_type(
                record["website"]
            )
        )

        record["category"] = (
            await extract_google_category(
                page,
                record["name"]
            )
        )

        record["latitude"], record["longitude"] = (
            extract_coordinates(
                record["maps_url"]
            )
        )

        record["place_id"] = (
            extract_place_id_from_url(
                record["maps_url"]
            ) or extract_place_id_from_url(url)
        )

        record["maps_internal_id"] = (
            extract_maps_internal_id(
                record["maps_url"]
            )
        )

        # Services
        services = (
            await extract_service_signals(
                page
            )
        )

        record["dine_in"] = (
            services["dine_in"]
        )

        record["takeaway"] = (
            services["takeaway"]
        )

        record["delivery"] = (
            services["delivery"]
        )

        # Status
        status = (
            await extract_business_status(
                page
            )
        )

        record["current_open_status"] = (
            status["current_open_status"]
        )

        record["business_status"] = (
            status["business_status"]
        )

        record["price_level"] = (
            await extract_price_level(
                page
            )
        )

        raw_hours = await extract_hours(
            page
        )

        record["hours"] = compress_hours(
            raw_hours
        )

        # Derived fields
        record["format"] = infer_format(
            record["category"],
            record["name"]
        )

        record["rough_brand"] = (
            conservative_brand(
                record["name"]
            )
        )

        record["data_confidence"] = (
            infer_confidence(
                record
            )
        )

        return record

    except Exception as exc:

        record["error"] = (
            f"{type(exc).__name__}: "
            f"{str(exc)[:500]}"
        )

        record["data_confidence"] = (
            "REVIEW: extraction exception"
        )

        return record

print("Shared place-detail extractor loaded.")


In [ ]:
# CELL 14 — Search collector

async def wait_for_search_state(page):
    # Wait for actual listing evidence or a terminal page, not just an empty feed shell.
    try:
        await page.wait_for_function(
            """() => {
                const text = document.body?.innerText || '';
                return document.querySelector('a[href*="/maps/place/"], a[href*="query_place_id="], a[href*="cid="], [role="feed"] [data-place-id], [role="feed"] [data-cid]') ||
                    (location.pathname.includes('/maps/place/') && document.querySelector('[role="main"] h1')) ||
                    /unusual traffic|automated queries|before you continue to google|enable javascript|no results found|google maps (?:can't|cannot) find/i.test(text);
            }""", timeout=SEARCH_READY_TIMEOUT_MS)
    except Exception:
        # Inspection below distinguishes timeout/unsupported layout from a real empty result.
        pass
    return await inspect_maps_page(page)


async def collect_search_results(page, query, max_results=30):
    if max_results <= 0:
        return []
    search_url = maps_search_url(query)
    print(f"\nSEARCH: {query}\n  URL: {search_url}")
    try:
        await safe_goto(page, search_url, wait_ms=1000)
        snapshot = await wait_for_search_state(page)
        if snapshot["state"] == "consent":
            await accept_google_consent(page)
            snapshot = await wait_for_search_state(page)
        if snapshot["state"] == "unrecognized":
            # One bounded reload can recover a stalled Maps JavaScript shell.
            print("  Maps did not render listings; retrying the search once.")
            await safe_goto(page, search_url, wait_ms=1000)
            snapshot = await wait_for_search_state(page)
        if snapshot["state"] == "no_results":
            append_log({"timestamp": now_iso(), "mode": "SEARCH_QUERY", "query": query, "url": page.url, "status": "NO_RESULTS"})
            print("  Google explicitly reported no matching results.")
            return []
        if snapshot["state"] not in {"results", "single_place"}:
            raise SearchCollectionError(f"Google Maps search did not load usable listings (state={snapshot['state']}).")
        if snapshot["state"] == "single_place":
            return [{"url": page.url, "label": snapshot["detail_name"]}]
        results = {}
        stagnant = 0
        for round_no in range(MAX_SEARCH_SCROLLS):
            # A results page can become blocked during scrolling; do not cache a partial success.
            if await detect_captcha(page):
                raise MapsAccessError("Google blocked the session during search scrolling.")
            before = len(results)
            for result in await discover_listing_links(page):
                results[normalize_maps_url(result["url"])] = result
            print(f"  scroll {round_no + 1:02d}: {len(results)} URLs")
            if len(results) >= max_results:
                break
            stagnant = stagnant + 1 if len(results) == before else 0
            if stagnant >= 4:
                break
            feed = page.locator('[role="feed"]').first
            if await feed.count():
                await feed.evaluate("el => { el.scrollTop = el.scrollHeight; }")
            else:
                await page.mouse.wheel(0, 5000)
            await page.wait_for_timeout(SEARCH_SCROLL_WAIT_MS)
        if not results:
            raise SearchCollectionError("The results layout was detected but no valid listing URLs could be read.")
        return list(results.values())[:max_results]
    except Exception as exc:
        report_path = await save_search_debug(page, query, f"{type(exc).__name__}: {exc}")
        append_log({"timestamp": now_iso(), "mode": "SEARCH_QUERY", "query": query, "url": page.url,
                    "status": "ERROR", "error": f"{type(exc).__name__}: {exc}; report={report_path}"})
        raise SearchCollectionError(f"SEARCH FAILED for {query!r}. See the screenshot above and {report_path}. Successful earlier observations remain in the checkpoint.") from exc

print("Search collector loaded (V2.3.3): empty failures now stop the run.")


In [ ]:

# CELL 15 — Target-area query generation from Google Sheets

def generate_queries(
    priority=TARGET_PRIORITY
):

    ws = spreadsheet.worksheet(
        "ISB-RWP Target Areas"
    )

    df = get_sheet_dataframe(
        ws
    )

    if df.empty:
        raise ValueError(
            "ISB-RWP Target Areas contains no data rows."
        )

    required = [
        "City",
        "Area/Sector",
        "Priority",
    ]

    missing = [
        c
        for c in required
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            "Target area sheet is missing: "
            + ", ".join(missing)
        )

    selected = df[
        df["Priority"].astype(str)
        .str.upper()
        .str.strip()
        == str(priority)
        .upper()
        .strip()
    ]

    queries = []

    for _, row in selected.iterrows():

        city = normalize_text(
            row["City"]
        )

        area_text = normalize_text(
            row["Area/Sector"]
        )

        if not city or not area_text:
            continue

        areas = [
            normalize_text(x)
            for x in re.split(
                r"[,;/|]",
                area_text
            )
            if normalize_text(x)
        ]

        for area in areas:

            for category in SEARCH_CATEGORIES:

                queries.append({
                    "city": city,
                    "area": area,
                    "category": category,
                    "priority": priority,
                    "query": (
                        f"{category} in "
                        f"{area} {city}"
                    ),
                })

    return queries


queries = generate_queries()

print(
    "Generated queries:",
    len(queries)
)

for item in queries[:20]:
    print(
        "-",
        item["query"]
    )


In [ ]:

# CELL 16 — Existing Restaurant Master reader from Google Sheets

def read_existing_master():

    ws = spreadsheet.worksheet(
        "Restaurant Master"
    )

    values = ws.get_all_values()

    if not values:
        return []

    df = clean_sheet_values(
        values
    )

    if df.empty:
        return []

    records = []

    # After removing blank rows, we cannot use
    # DataFrame index as an actual sheet row if blank
    # rows existed in the middle. Instead, scan the
    # original values to build actual row numbers.
    headers = [
        normalize_text(x)
        for x in values[0]
    ]

    for actual_row, raw_row in enumerate(
        values[1:],
        start=2
    ):

        padded = list(raw_row) + [
            ""
        ] * (
            len(headers)
            - len(raw_row)
        )

        if all(
            normalize_text(v) == ""
            for v in padded
        ):
            continue

        row_dict = {
            headers[i]: padded[i]
            for i in range(
                len(headers)
            )
            if headers[i]
        }

        records.append({
            "sheet_row": actual_row,
            "restaurant_id": row_dict.get(
                "Restaurant ID",
                ""
            ),
            "name": normalize_text(
                row_dict.get(
                    "Restaurant Name",
                    ""
                )
            ),
            "city": normalize_text(
                row_dict.get(
                    "City",
                    ""
                )
            ),
            "area": normalize_text(
                row_dict.get(
                    "Area",
                    ""
                )
            ),
            "maps_url": normalize_text(
                row_dict.get(
                    "Google Maps Link",
                    ""
                )
            ),
            **row_dict,
        })

    return records


existing_records = (
    read_existing_master()
)

print(
    "Existing Master rows:",
    len(existing_records)
)

def report_refresh_inputs(records):
    urls = [r for r in records if normalize_text(r.get("maps_url"))]
    print("Refresh input: ", len(records), "master rows;", len(urls), "rows with Maps links.")
    if not records:
        print("Restaurant Master has no data rows. REFRESH will be skipped; SEARCH can discover new restaurants.")
    elif not urls:
        print("No plain URLs were read from 'Google Maps Link'. Check that column and the selected Google Sheet; display labels/smart chips are not URLs.")
    return urls

report_refresh_inputs(existing_records)


In [ ]:
# CELL 17 — Drive checkpoint + log

def now_iso():
    return datetime.now(timezone.utc).isoformat()

def checkpoint_scope():
    return {"schema_version": 3, "spreadsheet_id": spreadsheet.id, "run_id": RUN_ID}

def load_checkpoint():
    scope = checkpoint_scope()
    if not RESET_CHECKPOINT and os.path.exists(CHECKPOINT_JSON):
        try:
            with open(CHECKPOINT_JSON, encoding="utf-8") as f:
                state = json.load(f)
            if all(state.get(k) == v for k, v in scope.items()) and isinstance(state.get("records"), dict) and isinstance(state.get("logs"), list):
                return state
        except (ValueError, OSError):
            print("Checkpoint unreadable; retaining it as a recovery file.")
        # Keep a previous run/corrupt checkpoint before starting fresh.
        os.replace(CHECKPOINT_JSON, CHECKPOINT_JSON + "." + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f") + ".bak")
    return {**scope, "created_at": now_iso(), "updated_at": now_iso(), "records": {}, "logs": []}

def save_checkpoint():
    checkpoint["updated_at"] = now_iso()
    temp_path = CHECKPOINT_JSON + ".tmp"
    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(checkpoint, f, ensure_ascii=False, indent=2, default=str)
    os.replace(temp_path, CHECKPOINT_JSON)

def checkpoint_key(mode, url):
    return f"{mode}|{normalize_maps_url(url)}"

def get_cached(mode, url):
    record = checkpoint["records"].get(checkpoint_key(mode, url))
    if not record or record.get("error") or not record.get("name"):
        return None
    try:
        age = (datetime.now(timezone.utc) - datetime.fromisoformat(record["observed_at"])).total_seconds()
        if not 0 <= age < CHECKPOINT_MAX_AGE_HOURS * 3600:
            return None
    except (KeyError, ValueError, TypeError):
        return None
    return dict(record)

def save_record_checkpoint(mode, url, record):
    checkpoint["records"][checkpoint_key(mode, url)] = dict(record)
    save_checkpoint()

def append_log(entry):
    checkpoint["logs"].append(entry)
    save_checkpoint()
    # Append across runs instead of replacing audit history.
    pd.DataFrame([entry]).reindex(columns=LOG_COLUMNS).to_csv(
        SCRAPE_LOG_CSV, mode="a", header=not os.path.exists(SCRAPE_LOG_CSV) or os.path.getsize(SCRAPE_LOG_CSV) == 0, index=False)

LOG_COLUMNS = ["timestamp", "mode", "restaurant_id", "restaurant", "url", "query", "city", "area", "category", "place_id", "rating", "review_count", "review_source", "hours", "dine_in", "takeaway", "delivery", "current_open_status", "business_status", "data_confidence", "status", "error"]
checkpoint = load_checkpoint()
print("Checkpoint/log system loaded.")

async def resumable_search_results(page, query, max_results):
    key = json.dumps([query, max_results], ensure_ascii=False)
    cached = checkpoint.setdefault("search_queries", {}).get(key)
    if cached:
        try:
            age = (datetime.now(timezone.utc) - datetime.fromisoformat(cached["observed_at"])).total_seconds()
            if 0 <= age < CHECKPOINT_MAX_AGE_HOURS * 3600:
                return cached["results"]
        except (KeyError, TypeError, ValueError):
            pass
    results = await collect_search_results(page, query, max_results)
    if results:  # Empty/failed searches are retried.
        checkpoint["search_queries"][key] = {"observed_at": now_iso(), "results": results}
        save_checkpoint()
    return results


In [ ]:

# CELL 18 — REFRESH mode

async def run_refresh(context):

    if not RUN_REFRESH:

        print(
            "REFRESH disabled."
        )

        return []

    candidates = [
        r
        for r in existing_records
        if r.get("maps_url")
    ]

    if MAX_REFRESH_RECORDS is not None:

        candidates = candidates[
            :MAX_REFRESH_RECORDS
        ]

    if not candidates:
        print("REFRESH skipped: no existing Maps URLs selected. SEARCH is independent of REFRESH.")
        return []
    page = await context.new_page()
    total = len(candidates)

    print(
        f"Refreshing {total} records..."
    )

    results = []

    for idx, base in enumerate(
        candidates,
        start=1
    ):

        url = base["maps_url"]

        cached = get_cached(
            "REFRESH",
            url
        )

        if cached:

            cached["existing_sheet_row"] = (
                base["sheet_row"]
            )

            cached["existing_restaurant_id"] = (
                base.get(
                    "Restaurant ID",
                    ""
                )
            )

            results.append(
                cached
            )

            print(
                f"[{idx}/{total}] "
                f"RESUME {base['name']} | "
                f"Reviews={cached.get('review_count')}"
            )

            continue

        started = now_iso()

        record = await extract_place_details(
            page,
            url=url,
            city=base.get(
                "City",
                ""
            ),
            area=base.get(
                "Area",
                ""
            ),
            search_category="",
            mode="REFRESH"
        )

        record[
            "existing_sheet_row"
        ] = base[
            "sheet_row"
        ]

        record[
            "existing_restaurant_id"
        ] = base.get(
            "Restaurant ID",
            ""
        )

        results.append(
            record
        )

        save_record_checkpoint(
            "REFRESH",
            url,
            record
        )

        append_log({
            "timestamp": started,
            "mode": "REFRESH",
            "restaurant_id": base.get(
                "Restaurant ID",
                ""
            ),
            "restaurant": base.get(
                "name",
                ""
            ),
            "url": url,
            "place_id": record.get(
                "place_id",
                ""
            ),
            "rating": record.get(
                "rating"
            ),
            "review_count": record.get(
                "review_count"
            ),
            "review_source": record.get(
                "review_source"
            ),
            "hours": record.get(
                "hours",
                ""
            ),
            "dine_in": record.get(
                "dine_in",
                ""
            ),
            "takeaway": record.get(
                "takeaway",
                ""
            ),
            "delivery": record.get(
                "delivery",
                ""
            ),
            "current_open_status": record.get(
                "current_open_status",
                ""
            ),
            "business_status": record.get(
                "business_status",
                ""
            ),
            "data_confidence": record.get(
                "data_confidence",
                ""
            ),
            "status": (
                "ERROR"
                if record.get("error")
                else "OK"
            ),
            "error": record.get(
                "error",
                ""
            ),
        })

        pct = round(
            idx / total * 100,
            1
        )

        print(
            f"[{idx}/{total} — {pct}%] "
            f"{record.get('name')} | "
            f"Rating={record.get('rating')} | "
            f"Reviews={record.get('review_count')} | "
            f"PlaceID={bool(record.get('place_id'))}"
        )

        await asyncio.sleep(
            random.uniform(
                MIN_DELAY,
                MAX_DELAY
            )
        )

    await page.close()

    return results


In [ ]:

# CELL 19 — SEARCH mode

async def run_search(context):

    if not RUN_SEARCH:

        print(
            "SEARCH disabled."
        )

        return []

    page = await context.new_page()

    search_plan = queries

    if MAX_SEARCH_QUERIES is not None:

        search_plan = search_plan[
            :MAX_SEARCH_QUERIES
        ]

    results = []

    for q_idx, query_info in enumerate(
        search_plan,
        start=1
    ):

        print(
            f"\nQUERY "
            f"{q_idx}/{len(search_plan)}"
        )

        search_results = (
            await resumable_search_results(
                page,
                query_info["query"],
                MAX_RESULTS_PER_QUERY
            )
        )

        for r_idx, result in enumerate(
            search_results,
            start=1
        ):

            url = result["url"]

            cached = get_cached(
                "SEARCH",
                url
            )

            if cached:

                cached["source_query"] = (
                    query_info["query"]
                )

                results.append(
                    cached
                )

                print(
                    f"  [{r_idx}/"
                    f"{len(search_results)}] "
                    f"RESUME "
                    f"{cached.get('name')} | "
                    f"Reviews="
                    f"{cached.get('review_count')}"
                )

                continue

            started = now_iso()

            # IMPORTANT:
            # SEARCH uses the exact same detail extractor
            # as REFRESH.
            record = (
                await extract_place_details(
                    page,
                    url=url,
                    city=query_info["city"],
                    area=query_info["area"],
                    search_category=query_info["category"],
                    mode="SEARCH"
                )
            )

            record[
                "source_query"
            ] = query_info[
                "query"
            ]

            results.append(
                record
            )

            save_record_checkpoint(
                "SEARCH",
                url,
                record
            )

            append_log({
                "timestamp": started,
                "mode": "SEARCH",
                "restaurant_id": "",
                "restaurant": record.get(
                    "name",
                    ""
                ),
                "url": url,
                "query": query_info["query"],
                "city": query_info["city"],
                "area": query_info["area"],
                "category": query_info["category"],
                "place_id": record.get(
                    "place_id",
                    ""
                ),
                "rating": record.get(
                    "rating"
                ),
                "review_count": record.get(
                    "review_count"
                ),
                "review_source": record.get(
                    "review_source"
                ),
                "hours": record.get(
                    "hours",
                    ""
                ),
                "dine_in": record.get(
                    "dine_in",
                    ""
                ),
                "takeaway": record.get(
                    "takeaway",
                    ""
                ),
                "delivery": record.get(
                    "delivery",
                    ""
                ),
                "current_open_status": record.get(
                    "current_open_status",
                    ""
                ),
                "business_status": record.get(
                    "business_status",
                    ""
                ),
                "data_confidence": record.get(
                    "data_confidence",
                    ""
                ),
                "status": (
                    "ERROR"
                    if record.get("error")
                    else "OK"
                ),
                "error": record.get(
                    "error",
                    ""
                ),
            })

            print(
                f"  [{r_idx}/"
                f"{len(search_results)}] "
                f"{record.get('name')} | "
                f"Rating="
                f"{record.get('rating')} | "
                f"Reviews="
                f"{record.get('review_count')} | "
                f"PlaceID="
                f"{bool(record.get('place_id'))}"
            )

            await asyncio.sleep(
                random.uniform(
                    MIN_DELAY,
                    MAX_DELAY
                )
            )

    await page.close()

    return results


In [ ]:

# CELL 20 — Run scraper

async def run_collection():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS, args=["--no-sandbox", "--disable-dev-shm-usage"])
        try:
            context = await browser.new_context(viewport=VIEWPORT, locale="en-US")
            refresh_results = await run_refresh(context)
            search_results = await run_search(context)
            return refresh_results, search_results
        finally:
            await browser.close()


refresh_results, search_results, all_records, unique_records = [], [], [], []
COLLECTION_STATUS = "RUNNING"
try:
    refresh_results, search_results = await run_collection()
except Exception:
    COLLECTION_STATUS = "FAILED"
    print("COLLECTION FAILED — Google Sheets sync is disabled. Check the diagnostic screenshot/report; earlier successful observations remain in the checkpoint.")
    raise
else:
    COLLECTION_STATUS = "COMPLETE"
    print("\nCOLLECTION COMPLETE")
    print("Refresh:", len(refresh_results))
    print("Search:", len(search_results))
    if not refresh_results and not search_results:
        print("No records collected. Check the REFRESH input summary and explicit search no-results logs above.")


In [ ]:

# CELL 21 — Deduplicate + diagnostics

def deduplicate_records(records):
    unique = {}
    for record in records:
        if record.get("error") or not normalize_text(record.get("name")):
            continue
        key = make_identity_key(record)
        existing = unique.get(key)
        if existing is None or record.get("observed_at", "") > existing.get("observed_at", ""):
            unique[key] = dict(record)
    for record in unique.values():
        record["data_confidence"] = infer_confidence(record)
    print("Collected:", len(records), "Successful unique:", len(unique))
    return list(unique.values())


all_records = (
    refresh_results
    + search_results
)

unique_records = deduplicate_records(
    all_records
)

RAW_RESULTS_JSON = os.path.join(OUTPUT_DIR, "v2_3_2_raw_results_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f") + ".json")
with open(
    RAW_RESULTS_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_records,
        f,
        ensure_ascii=False,
        indent=2,
        default=str
    )


def quality_report(records):

    total = len(records)

    if not total:
        print(
            "No records."
        )
        return

    fields = [
        ("Rating", "rating"),
        ("Review Count", "review_count"),
        ("Place ID", "place_id"),
        ("Maps URL", "maps_url"),
        ("Phone", "phone"),
        ("Address", "address"),
        ("Website", "website"),
        ("Hours", "hours"),
        ("Dine-In", "dine_in"),
        ("Takeaway", "takeaway"),
        ("Delivery", "delivery"),
        ("Google Category", "category"),
    ]

    print(
        "\n========== QUALITY =========="
    )

    print(
        "Total records:",
        total
    )

    for title, field in fields:

        filled = sum(
            1
            for r in records
            if r.get(field)
            not in (None, "")
        )

        pct = round(
            filled / total * 100,
            1
        )

        print(
            f"{title:18s} "
            f"{filled:4d}/{total} "
            f"({pct}%)"
        )

    print(
        "\nReview sources:"
    )

    print(
        pd.Series([
            r.get(
                "review_source",
                "unknown"
            )
            for r in records
        ]).value_counts(
            dropna=False
        ).to_string()
    )

    print(
        "\nConfidence:"
    )

    print(
        pd.Series([
            r.get(
                "data_confidence",
                "UNKNOWN"
            )
            for r in records
        ]).value_counts(
            dropna=False
        ).to_string()
    )


quality_report(
    unique_records
)


In [ ]:

# CELL 22 — Diagnostics table BEFORE Google Sheet write

def diagnostics_table(records):

    rows = []

    for r in records:

        rows.append({
            "Mode": r.get(
                "mode",
                ""
            ),
            "Restaurant": r.get(
                "name",
                ""
            ),
            "Rating": r.get(
                "rating"
            ),
            "Reviews": r.get(
                "review_count"
            ),
            "Review Source": r.get(
                "review_source",
                ""
            ),
            "Place ID": r.get(
                "place_id",
                ""
            ),
            "Maps Internal ID": r.get(
                "maps_internal_id",
                ""
            ),
            "Hours": r.get(
                "hours",
                ""
            ),
            "Dine-In": r.get(
                "dine_in",
                ""
            ),
            "Takeaway": r.get(
                "takeaway",
                ""
            ),
            "Delivery": r.get(
                "delivery",
                ""
            ),
            "Current Open": r.get(
                "current_open_status",
                ""
            ),
            "Business Status": r.get(
                "business_status",
                ""
            ),
            "Confidence": r.get(
                "data_confidence",
                ""
            ),
            "Error": r.get(
                "error",
                ""
            ),
        })

    diag = pd.DataFrame(
        rows
    )

    display(diag)

    return diag


diagnostics_df = diagnostics_table(
    unique_records
)


In [ ]:

# CELL 23 — Google Sheets writer helpers

def current_master_state():

    ws = spreadsheet.worksheet(
        "Restaurant Master"
    )

    values = ws.get_all_values()

    if not values:
        raise ValueError(
            "Restaurant Master is empty."
        )

    headers = [
        normalize_text(
            x
        )
        for x in values[0]
    ]

    return ws, values, headers


def ensure_v231_columns_gspread(ws, headers):
    headers = list(headers)
    missing = [c for c in V231_COLUMNS if c not in headers]
    if missing:
        needed = len(headers) + len(missing)
        if ws.col_count < needed:
            ws.add_cols(needed - ws.col_count)
        cells = [gspread.cell.Cell(row=1, col=len(headers) + i, value=c) for i, c in enumerate(missing, 1)]
        ws.update_cells(cells, value_input_option="RAW")
        headers.extend(missing)
    return {h: i for i, h in enumerate(headers, 1) if h}


def register_identity(maps, kind, key, row_number):
    if key:
        matches = maps[kind].setdefault(key, set())
        matches.add(row_number)
        if kind == "place_id" and len(matches) > 1:
            raise ValueError(f"Duplicate Place ID in master: {key}")


def master_identity_record(row):
    return {"place_id": row.get("Place ID") or extract_place_id_from_url(row.get("Google Maps Link", "")),
            "maps_url": row.get("Google Maps Link"), "phone": row.get("Phone Number"),
            "name": row.get("Restaurant Name"), "address": row.get("Full Address")}

def build_existing_identity_maps(ws, values, headers):
    maps = {"place_id": {}, "maps": {}, "phone": {}, "name_address": {}}
    for row_number, raw in enumerate(values[1:], 2):
        row = dict(zip(headers, list(raw) + [""] * (len(headers) - len(raw))))
        for kind, key in identity_entries(master_identity_record(row)):
            register_identity(maps, kind, key, row_number)
    return maps

def identity_entries(record):
    name, address = normalize_name(record.get("name")), normalize_name(record.get("address"))
    return [("place_id", normalize_text(record.get("place_id")) or extract_place_id_from_url(record.get("maps_url", ""))),
            ("maps", normalize_maps_url(record.get("maps_url"))),
            ("phone", phone_key(record.get("phone"))),
            ("name_address", f"{name}|{address}" if name and address else "")]


def next_restaurant_id_from_values(
    values,
    headers
):

    if "Restaurant ID" not in headers:
        return 1

    idx = headers.index(
        "Restaurant ID"
    )

    ids = []

    for raw_row in values[1:]:

        if idx >= len(raw_row):
            continue

        value = raw_row[idx]

        try:
            ids.append(
                int(
                    float(
                        value
                    )
                )
            )
        except Exception:
            pass

    return (
        max(ids) + 1
        if ids
        else 1
    )


def build_updates_for_record(
    record
):

    updates = {}

    mappings = [
        ("name", "Restaurant Name"),
        ("address", "Full Address"),
        ("phone", "Phone Number"),
        ("website", "Website URL"),
        ("city", "City"),
        ("area", "Area"),
    ]

    for source, target in mappings:

        value = record.get(
            source
        )

        if value not in (
            None,
            ""
        ):

            updates[
                target
            ] = value

    if record.get(
        "rough_brand"
    ):
        updates[
            "Brand Name"
        ] = record[
            "rough_brand"
        ]

    if record.get(
        "format"
    ):
        updates[
            "Format"
        ] = record[
            "format"
        ]

    if record.get(
        "latitude"
    ) is not None:

        updates[
            "Latitude"
        ] = record[
            "latitude"
        ]

    if record.get(
        "longitude"
    ) is not None:

        updates[
            "Longitude"
        ] = record[
            "longitude"
        ]

    if record.get(
        "hours"
    ):

        updates[
            "Operational Timing"
        ] = record[
            "hours"
        ]

    if record.get(
        "website_type"
    ):

        updates[
            "Website Type"
        ] = record[
            "website_type"
        ]

    for source, target in [
        ("dine_in", "Dine-In (Y/N)"),
        ("takeaway", "Takeaway (Y/N)"),
        ("delivery", "Delivery (Y/N)"),
    ]:

        if record.get(source) == "Y":

            updates[
                target
            ] = "Y"

    updates[
        "Google Business (Y/N)"
    ] = "Y"

    if record.get(
        "maps_url"
    ):

        updates[
            "Google Maps Link"
        ] = record[
            "maps_url"
        ]

    if record.get(
        "rating"
    ) is not None:

        updates[
            "Google Star Rating"
        ] = record[
            "rating"
        ]

    if record.get(
        "review_count"
    ) is not None:

        updates[
            "Google Review Count"
        ] = record[
            "review_count"
        ]

    updates[
        "Data Source"
    ] = "Google Maps"

    updates[
        "Date Profiled"
    ] = record.get("observed_at", "")[:10]

    if record.get(
        "foodpanda_listed"
    ) == "Y":

        updates[
            "Foodpanda Listed (Y/N)"
        ] = "Y"

    if record.get(
        "foodpanda_url"
    ):

        updates[
            "Foodpanda URL"
        ] = record[
            "foodpanda_url"
        ]

    return updates


In [ ]:
# CELL 24 — Google Sheets writer (single-writer workflow)

def locate_record_row(record, identity_maps):
    place_id = normalize_text(record.get("place_id")) or extract_place_id_from_url(record.get("maps_url", ""))
    entries = [("place_id", place_id)] if place_id else identity_entries(record)[1:]
    # With a Place ID there is deliberately no URL/phone/name fallback.
    for kind, key in entries:
        matches = identity_maps[kind].get(key, set())
        if len(matches) == 1:
            return next(iter(matches))
    return None

def validate_restaurant_ids(values, headers):
    index = headers.index("Restaurant ID")
    ids = set()
    for row in values[1:]:
        value = normalize_text(row[index] if index < len(row) else "")
        if not value:
            continue
        if not re.fullmatch(r"[1-9]\d*", value):
            raise ValueError(f"Restaurant ID must be a positive integer: {value}")
        number = int(value)
        if number in ids:
            raise ValueError(f"Duplicate Restaurant ID {number}; resolve existing identities before syncing.")
        ids.add(number)
    return max(ids, default=0) + 1

def write_to_google_sheet(records):
    if globals().get("COLLECTION_STATUS") in {"RUNNING", "FAILED"}:
        raise RuntimeError("Collection is incomplete/failed. Resolve the search error and rerun collection before syncing Google Sheets.")
    ws, values, headers = current_master_state()
    clean_sheet_values(values)  # Duplicate-header validation before mutation.
    if not all(headers):
        raise ValueError("Restaurant Master has blank headers; name every used column before syncing.")
    next_id = validate_restaurant_ids(values, headers)
    maps = build_existing_identity_maps(ws, values, headers)  # Reject duplicate Place IDs.
    # Snapshot is recovery evidence; do not overwrite previous pre-write states.
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f")
    with open(os.path.join(OUTPUT_DIR, f"master_before_write_{stamp}.json"), "w", encoding="utf-8") as f:
        json.dump({"spreadsheet_id": spreadsheet.id, "observed_at": now_iso(), "values": values}, f, ensure_ascii=False, indent=2)
    header_map = ensure_v231_columns_gspread(ws, headers)
    headers = list(header_map)
    rows = {i: dict(zip(headers, list(raw) + [""] * (len(headers) - len(raw)))) for i, raw in enumerate(values[1:], 2)}
    last_row = max((i for i, row in rows.items() if any(normalize_text(v) for v in row.values())), default=1)
    cell_updates = {}
    def set_value(row_number, column, value):
        if rows[row_number].get(column) != value:
            rows[row_number][column] = value
            cell_updates[(row_number, header_map[column])] = value
    # Assign missing IDs to every named restaurant, preserving all existing IDs.
    for row_number, row in rows.items():
        if row.get("Restaurant Name") and not normalize_text(row.get("Restaurant ID")):
            set_value(row_number, "Restaurant ID", str(next_id))
            next_id += 1
    added = updated = skipped = 0
    for record in records:
        if record.get("error") or not normalize_text(record.get("name")):
            skipped += 1
            continue
        target = locate_record_row(record, maps)
        is_new = target is None
        if is_new:
            last_row += 1
            target = last_row
            rows[target] = dict.fromkeys(headers, "")
            set_value(target, "Restaurant ID", str(next_id))
            next_id += 1
            set_value(target, "Status", "New Lead")
            added += 1
        else:
            updated += 1
            # Avoid regressing an already newer verified observation during recovery.
            previous = rows[target].get("Last Observed At", "")
            if previous and record.get("observed_at", "") < previous:
                skipped += 1
                continue
        old_entries = identity_entries(master_identity_record(rows[target]))
        updates = build_updates_for_record(record)
        # Remove legacy malformed schedules; retain previous validated hours if unavailable.
        existing_hours = rows[target].get("Operational Timing", "")
        if existing_hours and not is_valid_schedule(existing_hours):
            set_value(target, "Operational Timing", "")
        if updates.get("Operational Timing") and not is_valid_schedule(updates["Operational Timing"]):
            updates.pop("Operational Timing")
        for field, column in [("place_id", "Place ID"), ("category", "Google Category"),
                              ("current_open_status", "Current Open Status"), ("business_status", "Business Status"),
                              ("price_level", "Price Level"), ("data_confidence", "Data Confidence"),
                              ("observed_at", "Last Observed At")]:
            updates[column] = record.get(field)
        for column, value in updates.items():
            if column not in header_map or value in (None, ""):
                continue
            if column == "Business Status" and value == "Unknown" and rows[target].get(column):
                continue
            # Preserve analyst classifications/location and original profiling date.
            if column in {"Brand Name", "Format", "City", "Area", "Date Profiled"} and rows[target].get(column):
                continue
            set_value(target, column, value)
        # Update only this row's identity keys, including old URL/phone removal.
        for kind, key in old_entries:
            if key in maps[kind]:
                maps[kind][key].discard(target)
        for kind, key in identity_entries(master_identity_record(rows[target])):
            register_identity(maps, kind, key, target)
    if cell_updates:
        if last_row > ws.row_count:
            ws.add_rows(last_row - ws.row_count)
        # RAW protects phone prefixes and ensures scraped text cannot become a formula.
        cells = [gspread.cell.Cell(row=r, col=c, value=v) for (r, c), v in sorted(cell_updates.items())]
        ws.update_cells(cells, value_input_option="RAW")
    print(f"Google Sheet sync complete: {added} new, {updated} matched, {skipped} skipped.")
    return ws

write_to_google_sheet(unique_records)


In [ ]:

# CELL 25 — Repair Dashboard directly in Google Sheets

def repair_dashboard_google_sheet():

    try:

        dash = spreadsheet.worksheet(
            "Dashboard"
        )

        master = spreadsheet.worksheet(
            "Restaurant Master"
        )

    except Exception as exc:

        print(
            "Dashboard/master sheet not available:",
            exc
        )

        return

    master_headers = [
        normalize_text(x)
        for x in master.row_values(
            1
        )
    ]

    column_map = {
        header: idx + 1
        for idx, header
        in enumerate(
            master_headers
        )
        if header
    }

    # Convert numeric column numbers to letters.
    def col_letter(number):
        result = ""

        while number:
            number, rem = divmod(
                number - 1,
                26
            )

            result = (
                chr(
                    65 + rem
                )
                + result
            )

        return result

    range_end = 10000

    fields = {
        "Restaurant Name": "Restaurant Name",
        "City": "City",
        "POS Installed": "POS Installed (Y/N)",
        "No BI Team": "Dedicated BI Team (Y/N)",
        "Foodpanda Listed": "Foodpanda Listed (Y/N)",
    }

    if not all(
        field in column_map
        for field in fields.values()
    ):
        print(
            "Required master columns for dashboard "
            "repair are not all present."
        )

        return

    formulas = {
        "Total Restaurants": (
            f"=COUNTA('Restaurant Master'!"
            f"{col_letter(column_map['Restaurant Name'])}"
            f"2:{col_letter(column_map['Restaurant Name'])}"
            f"{range_end})"
        ),

        "Islamabad Count": (
            f'=COUNTIF(\'Restaurant Master\'!'
            f"{col_letter(column_map['City'])}"
            f"2:{col_letter(column_map['City'])}"
            f"{range_end},\"Islamabad\")"
        ),

        "Rawalpindi Count": (
            f'=COUNTIF(\'Restaurant Master\'!'
            f"{col_letter(column_map['City'])}"
            f"2:{col_letter(column_map['City'])}"
            f"{range_end},\"Rawalpindi\")"
        ),

        "POS Installed": (
            f'=COUNTIF(\'Restaurant Master\'!'
            f"{col_letter(column_map['POS Installed (Y/N)'])}"
            f"2:{col_letter(column_map['POS Installed (Y/N)'])}"
            f"{range_end},\"Y\")"
        ),

        "No BI Team": (
            f'=COUNTIF(\'Restaurant Master\'!'
            f"{col_letter(column_map['Dedicated BI Team (Y/N)'])}"
            f"2:{col_letter(column_map['Dedicated BI Team (Y/N)'])}"
            f"{range_end},\"N\")"
        ),

        "Foodpanda Listed": (
            f'=COUNTIF(\'Restaurant Master\'!'
            f"{col_letter(column_map['Foodpanda Listed (Y/N)'])}"
            f"2:{col_letter(column_map['Foodpanda Listed (Y/N)'])}"
            f"{range_end},\"Y\")"
        ),
    }

    dashboard_values = (
        dash.get_all_values()
    )

    updates = {}

    for row_number, row in enumerate(
        dashboard_values,
        start=1
    ):

        for col_number, value in enumerate(
            row,
            start=1
        ):

            label = normalize_text(
                value
            ).lower()

            for metric, formula in formulas.items():

                if (
                    metric.lower()
                    == label
                ):

                    # Put formula in the cell immediately
                    # to the right of the metric label.
                    updates[
                        (
                            row_number,
                            col_number + 1
                        )
                    ] = formula

    if updates:

        cells = [
            gspread.cell.Cell(
                row=row,
                col=col,
                value=value
            )
            for (
                row,
                col
            ), value in updates.items()
        ]

        dash.update_cells(
            cells, value_input_option="USER_ENTERED"
        )

        print(
            "Dashboard formulas repaired:",
            len(cells)
        )

    else:

        print(
            "No matching Dashboard metric labels found."
        )


if REPAIR_DASHBOARD:
    repair_dashboard_google_sheet()


In [ ]:

# CELL 26 — Final Google Sheet verification

def verify_google_sheet():

    ws = spreadsheet.worksheet(
        "Restaurant Master"
    )

    values = ws.get_all_values()

    if not values:
        print(
            "Restaurant Master is empty."
        )
        return

    headers = [
        normalize_text(x)
        for x in values[0]
    ]

    print(
        "\n========== FINAL SHEET QA =========="
    )

    print(
        "Rows:",
        max(0, len(values) - 1)
    )

    fields = [
        "Restaurant ID",
        "Restaurant Name",
        "Google Maps Link",
        "Google Star Rating",
        "Google Review Count",
        "Place ID",
        "Google Category",
        "Current Open Status",
        "Business Status",
        "Data Confidence",
        "Operational Timing",
        "Dine-In (Y/N)",
        "Takeaway (Y/N)",
        "Delivery (Y/N)",
        "Website URL",
        "Website Type",
    ]

    for field in fields:

        if field not in headers:

            print(
                field + ": COLUMN MISSING"
            )

            continue

        index = headers.index(
            field
        )

        filled = 0

        for row in values[1:]:

            if (
                index < len(row)
                and normalize_text(
                    row[index]
                ) != ""
            ):

                filled += 1

        print(
            f"{field:25s}: "
            f"{filled}/{len(values) - 1}"
        )

    # Duplicate checks.
    def nonblank_column(
        field
    ):

        if field not in headers:
            return []

        index = headers.index(
            field
        )

        return [
            normalize_text(
                row[index]
            )
            for row in values[1:]
            if (
                index < len(row)
                and normalize_text(
                    row[index]
                )
            )
        ]

    for field in [
        "Restaurant ID",
        "Place ID",
        "Google Maps Link",
    ]:

        data = nonblank_column(
            field
        )

        duplicates = (
            len(data)
            - len(set(data))
        )

        print(
            f"Duplicate {field}:",
            duplicates
        )


verify_google_sheet()


In [ ]:

# CELL 27 — Optional export of audit files already saved on Drive

print(
    "Drive audit files:"
)

for path in [
    CHECKPOINT_JSON,
    SCRAPE_LOG_CSV,
    RAW_RESULTS_JSON,
]:

    exists = os.path.exists(
        path
    )

    print(
        " -",
        path,
        "✅" if exists else "❌"
    )

print(
    "\nGoogle Sheet remains the system of record."
)

print("Search diagnostic folder:", os.path.join(OUTPUT_DIR, "search_debug"))
